# Data exploration

This notebook answers the Phase 3 exploration questions using the processed
match table produced by `02_data_cleaning.ipynb`. Run that notebook first (or
run `make data` then `02_data_cleaning.ipynb`) to regenerate
`data/processed/matches.csv` before executing this one.

In [1]:
from pathlib import Path

import pandas as pd

PROCESSED_PATH = Path("../data/processed/matches.csv")

matches = pd.read_csv(PROCESSED_PATH, parse_dates=["date"])
assert not matches.empty, "run 02_data_cleaning.ipynb before this notebook"
print(f"Loaded {len(matches)} processed matches from {PROCESSED_PATH}")

Loaded 49493 processed matches from ..\data\processed\matches.csv


## How many matches are in the dataset?

In [2]:
match_count = len(matches)
print(f"Match count: {match_count}")

Match count: 49493


## What years are covered?

In [3]:
date_min, date_max = matches["date"].min(), matches["date"].max()
print(f"Date range: {date_min.date()} to {date_max.date()}")

Date range: 1872-11-30 to 2026-07-03


## Which teams have played the most?

In [4]:
team_count = pd.unique(matches[["home_team", "away_team"]].values.ravel()).size
appearances = pd.concat([matches["home_team"], matches["away_team"]]).value_counts()
print(f"Distinct teams: {team_count}")
print(appearances.head(10))

Distinct teams: 336
Sweden         1105
England        1094
Argentina      1073
Brazil         1063
Germany        1035
South Korea    1010
Mexico         1007
Hungary        1006
Uruguay         973
France          939
Name: count, dtype: int64


## How many matches were played per year?

In [5]:
matches_by_year = matches["date"].dt.year.value_counts().sort_index()
print(matches_by_year.tail(15))

date
2012    1025
2013     963
2014     856
2015    1039
2016     920
2017     924
2018     929
2019    1149
2020     347
2021    1115
2022     970
2023    1054
2024    1231
2025    1002
2026     399
Name: count, dtype: int64


## What is the average number of goals per match?

In [6]:
avg_home_goals = matches["home_score"].mean()
avg_away_goals = matches["away_score"].mean()
avg_total_goals = (matches["home_score"] + matches["away_score"]).mean()
print(f"Average home goals: {avg_home_goals:.3f}")
print(f"Average away goals: {avg_away_goals:.3f}")
print(f"Average total goals per match: {avg_total_goals:.3f}")

Average home goals: 1.757
Average away goals: 1.182
Average total goals per match: 2.939


## What is the distribution of home wins, draws, and away wins?

In [7]:
outcome_distribution = matches["result"].value_counts(normalize=True).rename(
    {"H": "home_win", "D": "draw", "A": "away_win"}
)
print((outcome_distribution * 100).round(2).astype(str) + "%")

result
home_win    49.01%
away_win    28.25%
draw        22.74%
Name: proportion, dtype: object


## What are the most common scorelines?

In [8]:
scorelines = (
    matches["home_score"].astype(str) + "-" + matches["away_score"].astype(str)
).value_counts()
print(scorelines.head(10))

1-0    5104
1-1    4920
0-0    3972
2-0    3841
2-1    3780
0-1    3461
1-2    2552
3-0    2370
0-2    2229
2-2    1945
Name: count, dtype: int64


## Are there missing values or inconsistent team names?

In [9]:
missing_values = matches.isna().sum()
missing_values = missing_values[missing_values > 0]
print("Missing values by column:")
print(missing_values if not missing_values.empty else "None")

# An "inconsistent" team name would be one whose normalized form differs from
# its original upstream value in this snapshot.
renamed_home = (matches["home_team"] != matches["original_home_team"]).sum()
renamed_away = (matches["away_team"] != matches["original_away_team"]).sum()
print(f"Rows where normalization changed home_team: {renamed_home}")
print(f"Rows where normalization changed away_team: {renamed_away}")

Missing values by column:
None
Rows where normalization changed home_team: 0
Rows where normalization changed away_team: 0


### Validation: exploration summary sanity checks

In [10]:
assert match_count > 0
assert date_min <= date_max
assert team_count > 0
assert int(matches_by_year.sum()) == match_count
assert abs(outcome_distribution.sum() - 1.0) < 1e-9
assert avg_total_goals > 0
assert missing_values.empty, "processed matches must have no missing values"

print("Exploration summary validated.")

Exploration summary validated.
